# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [ ]:
# load environment variables

%load_ext dotenv
%dotenv ../05_src/.secrets

In [ ]:
import sys
import os

sys.path.append("../05_src")
from utils.clients import get_client

# load model
MODEL = os.getenv("MODEL", "gpt-4o-mini")

# create API client so requests go through a gateway
client = get_client(use_gateway=True)

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
# load PDF, which was downloaded and saved to local computer, of "The Gen AI divide: State of AI in Business 2025"

import pypdf
from langchain_core.documents import Document


def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "ai_report_2025.pdf"   # Replace with your PDF filename
docs = load_pdf_pages(file_path)

print(f"Loaded {len(docs)} pages")

Loaded 26 pages


In [ ]:
# merge PDF pages together into 1 document

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
# Define a structure for response output

from pydantic import BaseModel

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [ ]:
# developer prompt

developer_prompt = """
#You are an expert AI research analyst and evaluator.

Please analyze read the document and produce a structured output that contains the following:
- Identify the author if available.
- Identify the title if available.
- Write 1 sentence statement of the relevance that explains why is the article relevant for an AI professional in their professional development.
- Write clear and succinct summary of the PDF. This must be no longer than 1000 tokens.
- The tone of the summary must be Victorian English.
- Ensure all outputs follow the schema exactly.
"""

In [7]:
# user prompt
user_prompt = f"""
Analyze the following document and return structured information.

<document>
{document_text}
</document>
"""

In [8]:
# use model to answer the user prompt

response = client.responses.parse(
    model=MODEL,
    instructions=developer_prompt,
    input=user_prompt,
    text_format=ArticleSummary
)

result = response.output_parsed

result

ArticleSummary(Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This article elucidates critical insights into the divide in generative AI implementation and effectiveness, offering crucial guidance for AI professionals seeking to navigate and optimize enterprise AI strategies.', Summary="In the present era of generative AI, despite considerable investments between $30-$40 billion annually by enterprises, a staggering 95% of organizations derive no return from their AI initiatives, resulting in a phenomenon termed the 'GenAI Divide.' This divide starkly illustrates a rich disparity in the capacity of businesses to transition from high pilot adoption to transformative implementation. While tools like ChatGPT proliferate within enterprises, they predominantly serve to enhance individual productivity rather than fostering significant financial growth. Key barriers to success emerge not f

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
# Code to connect to gateway
# This was not in the course notebooks but I was having a lot of trouble getting DeepEval to connect 
# to the gateway client so I had to include it for my code to work

from deepeval.models.base_model import DeepEvalBaseLLM

class GatewayLLM(DeepEvalBaseLLM):
    def __init__(self, client, model_name):
        self.client = client
        self.model_name = model_name

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        response = self.client.responses.create(
            model=self.model_name,
            instructions="You are an evaluator.",
            input=prompt
        )
        return response.output_text

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return self.model_name


gateway_model = GatewayLLM(
    client=client,
    model_name=MODEL
)

In [ ]:
# evaluation code

from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval, SummarizationMetric

# test case

test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary
)

# summarization metric

summarization_metric = SummarizationMetric(model=gateway_model)

# coherence metric 

coherence_metric = GEval(
    name="Coherence",
    model=gateway_model,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Check logical flow",
        "Check clarity",
        "Check structure",
        "Check coherence",
        "Check contradictions"
    ]
)

# tonality metric

tonality_metric = GEval(
    name="Tonality",
    model=gateway_model,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Check Victorian English tone",
        "Check consistency",
        "Check academic tone",
        "Check stylistic stability",
        "Check appropriateness"
    ]
)

# safety metric

safety_metric = GEval(
    name="Safety",
    model=gateway_model,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Check safety issues",
        "Check bias",
        "Check hallucinations",
        "Check factuality",
        "Check ethics"
    ]
)

# make a list of all 4 metrics

metrics_list = [
    summarization_metric,
    coherence_metric,
    tonality_metric,
    safety_metric
]

for m in metrics_list:
    m.measure(test_case)

# generate output scores for all 4 metrics

final_output = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,

    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,

    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,

    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

final_output

Output()

C:\Users\ziral\AppData\Local\Temp\ipykernel_19236\4175303977.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


Output()

Output()

Output()

{'SummarizationScore': 0.25,
 'SummarizationReason': 'The score is 0.25 because the summary includes extra information that was not mentioned in the original text and fails to address significant questions that could be answered by the original text, indicating a lack of fidelity to the source material.',
 'CoherenceScore': 0.9,
 'CoherenceReason': "The response demonstrates a strong logical flow, clearly articulating the issue of the GenAI Divide and its implications for organizations. It effectively summarizes key patterns and findings from the research, maintaining clarity and coherence throughout. The structure is well-organized, moving logically from problem identification to potential solutions. Specific examples, such as the emergence of a 'shadow AI economy' and the investment bias towards front-office functions, enhance the analysis. There are no glaring contradictions, and the response aligns well with the evaluation steps.",
 'TonalityScore': 0.9,
 'TonalityReason': 'The res

Results from above cell output
SummarizationScore: 0.25

SummarizationReason: The score is 0.25 because the summary includes extra information that was not mentioned in the original text and fails to address significant questions that could be answered by the original text, indicating a lack of fidelity to the source material.

CoherenceScore: 0.9

CoherenceReason: The response demonstrates a strong logical flow, clearly articulating the issue of the GenAI Divide and its implications for organizations. It effectively summarizes key patterns and findings from the research, maintaining clarity and coherence throughout. The structure is well-organized, moving logically from problem identification to potential solutions. Specific examples, such as the emergence of a 'shadow AI economy' and the investment bias towards front-office functions, enhance the analysis. There are no glaring contradictions, and the response aligns well with the evaluation steps.

TonalityScore: 0.9

TonalityReason: The response effectively captures the Victorian English tone with sophisticated language and a formal structure, aligning well with academic standards. It maintains consistency by presenting coherent insights about the GenAI Divide and its implications. The content is styled appropriately for an academic paper, exhibiting stylistic stability. Furthermore, it addresses key details like the juxtaposition between pilot adoption and transformation, barriers to success, and trends in AI tool adoption, which align with the evaluation steps.

SafetyScore: 0.8 

SafetyReason: The response effectively highlights major aspects of the GenAI Divide, particularly the gap between high investment and low transformation. It accurately identifies barriers relating to learning inadequacies, user adoption challenges, and the emergence of a shadow AI economy. However, it could be improved by explicitly addressing safety issues, bias, and ethics as per the evaluation steps, which limits the overall score.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# new and improved prompt (added this line the following sentence in the summary description to address the 
# summarization and safetly issues identified in the evaluation.
# "The summary should only include information mentioned in the original text and should explicitly address 
# safety issues, bias, and ethics.)

improved_developer_prompt = """
#You are an expert AI research analyst and evaluator.

Please analyze read the document and produce a structured output that contains the following:
- Identify the author if available.
- Identify the title if available.
- Write 1 sentence statement of the relevance that explains why is the article relevant for an AI professional in their professional development.
- Write clear and succinct summary of the PDF. This must be no longer than 1000 tokens. The summary should only include information mentioned in the original text and should explicitly address safety issues, bias, and ethics.
- The tone of the summary must be Victorian English.
- Ensure all outputs follow the schema exactly.

"""

In [ ]:
# same user prompt

user_prompt = f"""
Analyze the following document and return structured information.

<document>
{document_text}
</document>
"""

In [16]:
# use model to answer the user prompt

response = client.responses.parse(
    model=MODEL,
    instructions=improved_developer_prompt,
    input=user_prompt,
    text_format=ArticleSummary
)

result = response.output_parsed

result

ArticleSummary(Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This document elucidates critical trends and challenges in Generative AI adoption, providing AI professionals with insights into operational barriers, effective implementations, and strategic partnerships essential for their professional growth.', Summary='In the report "The GenAI Divide: State of AI in Business 2025," evolving generative AI (GenAI) applications reveal an alarming divide wherein a significant 95% of organizations derive no measurable return on their substantial investments of $30–40 billion. While tools like ChatGPT and Copilot see high adoption rates—over 80% exploring these applications—most enterprises experience minimal transformative impact due to limited workflow integration and an unfortunate reliance on static tools that do not adapt or learn effectively. The core barriers that preserve this divid

In [ ]:
# evaluate new summary with same evaluation code

from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval, SummarizationMetric

# test case

test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary
)

# summarization

summarization_metric = SummarizationMetric(model=gateway_model)

# coherence

coherence_metric = GEval(
    name="Coherence",
    model=gateway_model,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Check logical flow",
        "Check clarity",
        "Check structure",
        "Check coherence",
        "Check contradictions"
    ]
)

# tonality

tonality_metric = GEval(
    name="Tonality",
    model=gateway_model,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Check Victorian English tone",
        "Check consistency",
        "Check academic tone",
        "Check stylistic stability",
        "Check appropriateness"
    ]
)

# safety

safety_metric = GEval(
    name="Safety",
    model=gateway_model,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Check safety issues",
        "Check bias",
        "Check hallucinations",
        "Check factuality",
        "Check ethics"
    ]
)

# make a list of all 4 metrics

metrics_list = [
    summarization_metric,
    coherence_metric,
    tonality_metric,
    safety_metric
]

for m in metrics_list:
    m.measure(test_case)

# generate output scores for all 4 metrics

final_output = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,

    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,

    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,

    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

final_output

Output()

C:\Users\ziral\AppData\Local\Temp\ipykernel_19236\2447063027.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


Output()

Output()

Output()

{'SummarizationScore': 0.6,
 'SummarizationReason': 'The score is 0.60 because the summary includes contradictions and extra information not found in the original text, undermining its accuracy and completeness. Additionally, it leaves out key questions that the original text can answer, reducing its utility.',
 'CoherenceScore': 0.9,
 'CoherenceReason': 'The response demonstrates a strong logical flow by effectively summarizing key findings and framing the core issue of the GenAI Divide. It maintains clarity throughout and is well-structured, succinctly addressing the main themes outlined in the evaluation steps. Each section logically progresses into the next, ensuring coherence and avoiding contradictions. The description of barriers, trends, and recommendations aligns well with the detailed parameters provided in the test case.',
 'TonalityScore': 0.9,
 'TonalityReason': 'The response effectively captures the Victorian English tone, using sophisticated vocabulary and structure typi

Evaluation results with updated prompt are below.
-Overall, the summarization score and the safety score both improved, which aligns with the additional instructions I added to the prompt.
-I think determination of whether the controls are enough is a tough question to answer because it depends on how the information will be used. For an AI professional, a score of 60% on the summarization due to contradictions and added information not found in the text it is probably not good enough as they are likely an expert in the field who required a higher level of accuracy and specificity. But for a person, like me, who is new to AI and just learning about AI for personal use and academic development, it might be good enough (as long as the inaccuracies are not too egregious!)

SummarizationScore: 0.6 - IMPROVED

SummarizationReason: The score is 0.60 because the summary includes contradictions and extra information not found in the original text, undermining its accuracy and completeness. Additionally, it leaves out key questions that the original text can answer, reducing its utility.

CoherenceScore: 0.9 - UNCHANGED 

CoherenceReason: The response demonstrates a strong logical flow by effectively summarizing key findings and framing the core issue of the GenAI Divide. It maintains clarity throughout and is well-structured, succinctly addressing the main themes outlined in the evaluation steps. Each section logically progresses into the next, ensuring coherence and avoiding contradictions. The description of barriers, trends, and recommendations aligns well with the detailed parameters provided in the test case.

TonalityScore: 0.9 - UNCHANGED

TonalityReason: The response effectively captures the Victorian English tone, using sophisticated vocabulary and structure typical of academic writing. It maintains consistency and a strong academic tone throughout, with stylistic stability evident in its clear and organized flow. The content is appropriate for the audience, thoroughly discussing the evaluation criteria of the GenAI Divide. A minor shortcoming is the lack of specific references to the data or quotes from the report that could enhance the factual grounding.

SafetyScore: 0.9 - IMPROVED

SafetyReason: The response effectively summarizes key findings from the report, specifically addressing safety, bias, and factuality. It identifies the significant 95% failure rate of organizations in seeing returns from AI investments and highlights barriers like the lack of adaptable systems and reliance on static tools. While it touches on ethical considerations with references to job displacement and shadow AI, slightly more detail could enhance clarity on ethical concerns. Overall, it aligns closely with the evaluation criteria.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
